<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/03_encoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.6.0?labpath=examples%2F03_encoders.ipynb)

# 03 — Encoders Compared

Side-by-side look at all three Phase 1 encoders:

| Encoder | Strategy | Qubits for d features |
|---|---|---|
| `AngleEncoder` | One rotation gate per feature | d |
| `AmplitudeEncoder` | State vector \|ψ⟩ = Σ xᵢ\|i⟩ | ⌈log₂ d⌉ |
| `BasisEncoder` | Binarize → X gate on \|1⟩ qubits | d |

```bash
pip install quprep
```

In [ ]:
import numpy as np

from quprep.encode.amplitude import AmplitudeEncoder
from quprep.encode.angle import AngleEncoder
from quprep.encode.basis import BasisEncoder
from quprep.export.qasm_export import QASMExporter

exporter = QASMExporter()

## Shared input

In [ ]:
feature_vector = np.array([0.1, 0.9, 0.4, 0.6])
print(f"Feature vector: {feature_vector}")

## 1. AngleEncoder

Each feature xᵢ becomes a rotation angle θᵢ on qubit i.

$$|\psi\rangle = \bigotimes_{i=1}^{d} R_y(\theta_i)|0\rangle$$

- Ry expects angles in **[0, π]**
- Rx/Rz expect angles in **[−π, π]**
- The Pipeline's auto-normalizer handles this for you

In [ ]:
angle_input = feature_vector * np.pi     # manually scale to [0, π] for illustration
enc_angle = AngleEncoder(rotation="ry").encode(angle_input)

print(f"Parameters : {enc_angle.parameters.round(4)}")
print(f"Metadata   : {enc_angle.metadata}")
print()
print(exporter.export(enc_angle))

## 2. AmplitudeEncoder

Encodes 2ⁿ features into n qubits using the quantum state vector.

$$|\psi\rangle = \sum_{i=0}^{2^n - 1} x_i |i\rangle, \quad \|\mathbf{x}\|_2 = 1$$

- Input **must** be unit-norm
- Dimension is automatically padded to the next power of two

In [ ]:
unit_vec = feature_vector / np.linalg.norm(feature_vector)
enc_amp = AmplitudeEncoder().encode(unit_vec)

print(f"Input      : {unit_vec.round(4)}")
print(f"‖x‖        : {np.linalg.norm(unit_vec):.6f}")
print(f"Parameters : {enc_amp.parameters.round(4)}")
print(f"Metadata   : {enc_amp.metadata}")
print()
print("QASM export is not supported for amplitude encoding.")
print("Use QiskitExporter — it uses Qiskit's StatePreparation gate.")

### Auto-padding: 5 features → 8 (next power of two = 3 qubits)

In [ ]:
vec5 = np.array([0.4, 0.3, 0.5, 0.6, 0.4])
vec5 = vec5 / np.linalg.norm(vec5)
enc5 = AmplitudeEncoder(pad=True).encode(vec5)

print("Input length  : 5")
print(f"Padded length : {len(enc5.parameters)}")
print(f"n_qubits      : {enc5.metadata['n_qubits']}")
print(f"‖padded vec‖  : {np.linalg.norm(enc5.parameters):.6f}")

## 3. BasisEncoder

Binarizes each feature at a threshold and applies X gates for qubits where the bit is 1.

$$|\psi\rangle = \bigotimes_{i=1}^{d} |b_i\rangle, \quad b_i = \mathbf{1}[x_i > \text{threshold}]$$

- Very simple and hardware-efficient
- Best for already-binary or easily thresholded features

In [ ]:
enc_basis = BasisEncoder(threshold=0.5).encode(feature_vector)

print(f"Input      : {feature_vector}")
print("Threshold  : 0.5")
print(f"Binary out : {enc_basis.parameters}")
print(f"Metadata   : {enc_basis.metadata}")
print()
print(exporter.export(enc_basis))

## 4. Comparison summary

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        "Encoder": "AngleEncoder (Ry)",
        "n_qubits": enc_angle.metadata["n_qubits"],
        "parameters": list(enc_angle.parameters.round(3)),
        "QASM support": "✓",
        "Qiskit support": "✓",
    },
    {
        "Encoder": "AmplitudeEncoder",
        "n_qubits": enc_amp.metadata["n_qubits"],
        "parameters": list(enc_amp.parameters.round(3)),
        "QASM support": "✗",
        "Qiskit support": "✓",
    },
    {
        "Encoder": "BasisEncoder",
        "n_qubits": enc_basis.metadata["n_qubits"],
        "parameters": list(enc_basis.parameters),
        "QASM support": "✓",
        "Qiskit support": "✓",
    },
])
summary